# Phase 23 — economic interpretation of multiscale regimes

This notebook builds state cards for the model selected by the frozen phase-22 validation rule. Economic interpretation means describing current hierarchy, persistence, activity, and later diagnostic behaviour. It does not create entries, exits, position direction, or a backtest.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from dc_multiscale_research import (
    FeatureTier,
    ModelFamily,
    ObservationClock,
    REGIME_ORDER,
    add_future_diagnostics,
    build_multiscale_state,
    candidate_summary,
    filtering_prefix_invariant,
    fit_candidate,
    load_close,
    plot_price_states,
    prepare_data,
    ratio_ladder,
    transition_matrix,
)

DATA_ROOT = Path(r"D:\data_dump\market_data\raw\FXPIG-Server\5")
SYMBOL = "GBPUSD.r"
THRESHOLDS = ratio_ladder(0.002, 1.5, 5)
CLOCK = ObservationClock.UNION_EVENT
FEATURE_TIER = FeatureTier.DYNAMIC
STATE_COUNTS = (3, 4, 5)
HMM_FAMILIES = (ModelFamily.NORMAL_HMM, ModelFamily.STUDENTT_HMM)
RANDOM_STATE = 17
MAX_ITER = 50
N_RESTARTS = 3
MINIMUM_VALIDATION_OCCUPANCY = 0.02
FUTURE_HORIZONS = (6, 24, 96)
LAST_N = 8_000

## Reproduce the frozen validation selection

No economic profile participates in model selection. The family and state count are chosen using validation likelihood, prefix causality, and the occupancy floor before state cards are constructed.

In [2]:
prices = load_close(DATA_ROOT, SYMBOL)
_, global_state, _ = build_multiscale_state(prices, THRESHOLDS, include_scale_state=False)
prepared = prepare_data(global_state, CLOCK, FEATURE_TIER)

fits = []
selection_records = []
for family in HMM_FAMILIES:
    for n_states in STATE_COUNTS:
        fit = fit_candidate(
            prepared, family, n_states, RANDOM_STATE, max_iter=MAX_ITER, n_restarts=N_RESTARTS
        )
        fits.append(fit)
        record = candidate_summary(fit, prepared)
        prefix_pass = filtering_prefix_invariant(fit, prepared)
        record["prefix_pass"] = prefix_pass
        record["eligible"] = (
            fit.error is None
            and prefix_pass is True
            and record.get("minimum_state_occupancy", 0.0) >= MINIMUM_VALIDATION_OCCUPANCY
        )
        selection_records.append(record)
selection_table = pd.DataFrame.from_records(selection_records)
eligible = selection_table.loc[selection_table["eligible"]]
if eligible.empty:
    raise RuntimeError("No eligible HMM candidate; inspect selection_table.")
selected_row = eligible.sort_values("validation_log_likelihood_per_row", ascending=False).iloc[0]
selected_fit = next(
    fit for fit in fits
    if fit.family.value == selected_row["family"] and fit.n_states == int(selected_row["n_states"])
)
display(selection_table.set_index(["family", "n_states"]))
print(f"Interpretation model: {selected_fit.family.value}, K={selected_fit.n_states}")

clock     tier error  \
family       n_states                                    
normal_hmm   3         union_event_time  dynamic  None   
             4         union_event_time  dynamic  None   
             5         union_event_time  dynamic  None   
studentt_hmm 3         union_event_time  dynamic  None   
             4         union_event_time  dynamic  None   
             5         union_event_time  dynamic  None   

                       validation_log_likelihood_per_row  \
family       n_states                                      
normal_hmm   3                                -25.503625   
             4                                -23.770640   
             5                                -15.796295   
studentt_hmm 3                                -29.261737   
             4                                -18.936978   
             5                                -10.867945   

                       minimum_state_occupancy  switch_rate  median_dwell  \
family       n_states                                                       
normal_hmm   3                        0.308797     0.214029           4.0   
             4                        0.109515     0.329137           2.0   
             5                        0.109515     0.750000           1.0   
studentt_hmm 3                        0.308797     0.214029           4.0   
             4                        0.134650     0.570144           1.0   
             5                        0.129264     0.591727           1.0   

                       mean_max_probability  baseline_adjusted_rand  \
family       n_states                                                 
normal_hmm   3                     1.000000                0.466753   
             4                     0.997412                0.349851   
             5                     0.997405                0.784826   
studentt_hmm 3                     0.999987                0.466753   
             4                     1.000000                0.296693   
             5                     1.000000                0.482655   

                       baseline_nmi  baseline_mapped_agreement  prefix_pass  \
family       n_states                                                         
normal_hmm   3             0.568454                   0.651706         True   
             4             0.520404                   0.542190         True   
             5             0.879244                   0.791741         True   
studentt_hmm 3             0.568454                   0.651706         True   
             4             0.438379                   0.500898         True   
             5             0.603379                   0.630162         True   

                       eligible  
family       n_states            
normal_hmm   3             True  
             4             True  
             5             True  
studentt_hmm 3             True  
             4             True  
             5             True

Interpretation model: studentt_hmm, K=5


## Construct one causal state path

Raw model states are retained. Training-only semantic names are annotations that explain strongest overlap with the direct taxonomy; they do not redefine the fitted states.

In [3]:
state_parts = []
for split_name in ("train", "validation", "test"):
    part = prepared.frames[split_name].copy()
    part["partition"] = split_name
    part["model_state"] = selected_fit.states[split_name]
    part["model_semantic"] = [REGIME_ORDER[code] for code in selected_fit.mapped_regimes[split_name]]
    part["filtered_confidence"] = selected_fit.probabilities[split_name].max(axis=1)
    for state in range(selected_fit.n_states):
        part[f"filtered_p_state_{state}"] = selected_fit.probabilities[split_name][:, state]
    state_parts.append(part)
state_path = pd.concat(state_parts).sort_index()
if not state_path["feature_information_time"].eq(state_path["time"]).all():
    raise AssertionError("A state-path feature timestamp differs from its observation time.")
display(state_path.tail(10))

,time,feature_information_time,bar_position,price,threshold_count,configuration_code,configuration,broad_agreement,coherent_frontier,direction_boundaries,...,log_completed_r_04,partition,model_state,model_semantic,filtered_confidence,filtered_p_state_0,filtered_p_state_1,filtered_p_state_2,filtered_p_state_3,filtered_p_state_4
time,,,,,,,,,,,,,,,,,,,,,
2026-11-05 13:05:00+02:00,2026-11-05 13:05:00+02:00,2026-11-05 13:05:00+02:00,257991,1.34582,5,28,down/down/up/up/up,False,True,1,...,-11.589347,test,2,shallow_counter,1.0,2.783215e-51,1.786429e-28,1.000000e+00,1.444056e-85,1.089326e-92
2026-11-05 21:15:00+02:00,2026-11-05 21:15:00+02:00,2026-11-05 21:15:00+02:00,258089,1.34549,5,28,down/down/up/up/up,False,True,1,...,-11.589347,test,2,shallow_counter,1.0,6.097769e-53,3.640256e-30,1.000000e+00,3.858127e-80,5.075353e-92
2026-11-06 05:15:00+02:00,2026-11-06 05:15:00+02:00,2026-11-06 05:15:00+02:00,258185,1.34374,5,24,down/down/down/up/up,False,True,1,...,-11.589347,test,2,shallow_counter,1.0,1.523608e-58,6.516907e-18,1.000000e+00,1.166244e-79,4.577036e-77
2026-11-06 09:00:00+02:00,2026-11-06 09:00:00+02:00,2026-11-06 09:00:00+02:00,258230,1.34022,5,16,down/down/down/down/up,False,True,1,...,-11.589347,test,1,deep_counter,1.0,1.633957e-67,1.000000e+00,1.351812e-17,1.030877e-63,7.364103e-52
2026-11-06 15:25:00+02:00,2026-11-06 15:25:00+02:00,2026-11-06 15:25:00+02:00,258307,1.33636,5,0,down/down/down/down/down,True,True,0,...,-11.386649,test,4,broad_coherence,1.0,8.813009e-63,5.731502e-48,1.267073e-66,2.851896e-64,1.000000e+00
2026-11-06 17:20:00+02:00,2026-11-06 17:20:00+02:00,2026-11-06 17:20:00+02:00,258330,1.33768,5,1,up/down/down/down/down,False,True,1,...,-11.386649,test,3,fragmented,1.0,8.395535e-118,4.094994e-75,1.539470e-90,1.000000e+00,3.963212e-85
2026-11-06 21:45:00+02:00,2026-11-06 21:45:00+02:00,2026-11-06 21:45:00+02:00,258383,1.33490,5,0,down/down/down/down/down,True,True,0,...,-11.386649,test,4,broad_coherence,1.0,1.567092e-64,2.051450e-49,1.445727e-128,2.606430e-42,1.000000e+00
2026-11-09 03:15:00+02:00,2026-11-09 03:15:00+02:00,2026-11-09 03:15:00+02:00,258449,1.33713,5,1,up/down/down/down/down,False,True,1,...,-11.386649,test,3,fragmented,1.0,5.261938e-114,1.860455e-70,5.887166e-92,1.000000e+00,1.751119e-79
2026-11-09 08:10:00+02:00,2026-11-09 08:10:00+02:00,2026-11-09 08:10:00+02:00,258508,1.33851,5,3,up/up/down/down/down,False,True,1,...,-11.386649,test,3,fragmented,1.0,4.993897e-64,1.916535e-26,5.398350e-108,1.000000e+00,2.922090e-51


## State cards: current economic structure

These fields answer what the state means **now**: direction balance, hierarchy coherence, scale reach, live reversal pressure, event activity, confidence, and direct-regime overlap.

In [4]:
current_fields = [
    "fragmentation",
    "counter_depth_fraction",
    "signed_mode_balance",
    "mean_mode_age_bars",
    "mean_reversal_progress",
    "max_reversal_progress",
    "hierarchy_dcc_rate",
    "filtered_confidence",
]
state_cards = state_path.groupby("model_state", observed=True)[current_fields].agg(
    ["count", "median", "mean"]
)
baseline_overlap = pd.crosstab(
    state_path["model_state"], state_path["structural_regime"], normalize="index"
).reindex(columns=REGIME_ORDER, fill_value=0.0)
direction_overlap = pd.crosstab(
    state_path["model_state"], state_path["outer_mode"], normalize="index"
)
display(state_cards)
display(baseline_overlap.style.format("{:.2%}"))
display(direction_overlap.style.format("{:.2%}"))

fragmentation              counter_depth_fraction         \
                    count median  mean                  count median   
model_state                                                            
0                     416   0.00  0.00                    416    0.0   
1                     442   0.25  0.25                    442    0.8   
2                     965   0.25  0.25                    965    0.4   
3                     919   0.25  0.25                    919    0.4   
4                     382   0.00  0.00                    382    0.0   

                      signed_mode_balance                  mean_mode_age_bars  \
                 mean               count median      mean              count   
model_state                                                                     
0            0.000000                 416    1.0  1.000000                416   
1            0.800000                 442   -0.6 -0.048869                442   
2            0.372228                 965    0.2  0.255544                965   
3            0.374755                 919   -0.2 -0.250490                919   
4            0.000000                 382   -1.0 -1.000000                382   

             ... mean_reversal_progress max_reversal_progress            \
             ...                   mean                 count    median   
model_state  ...                                                          
0            ...               0.039389                   416  0.073325   
1            ...               0.166277                   442  0.702660   
2            ...               0.317633                   965  0.706696   
3            ...               0.315762                   919  0.705446   
4            ...               0.040590                   382  0.076565   

                      hierarchy_dcc_rate                      \
                 mean              count    median      mean   
model_state                                                    
0            0.097738                416  0.005000  0.005431   
1            0.733716                442  0.005000  0.005958   
2            0.733231                965  0.004167  0.004985   
3            0.730415                919  0.005000  0.005385   
4            0.099454                382  0.005000  0.005406   

            filtered_confidence              
                          count median mean  
model_state                                  
0                           416    1.0  1.0  
1                           442    1.0  1.0  
2                           965    1.0  1.0  
3                           919    1.0  1.0  
4                           382    1.0  1.0  

[5 rows x 24 columns]

structural_regime,broad_coherence,shallow_counter,deep_counter,fragmented
model_state,,,,
0,100.00%,0.00%,0.00%,0.00%
1,0.00%,0.00%,100.00%,0.00%
2,0.00%,72.64%,27.36%,0.00%
3,0.00%,72.58%,27.42%,0.00%
4,100.00%,0.00%,0.00%,0.00%


outer_mode,down,up
model_state,,
0,0.00%,100.00%
1,45.93%,54.07%
2,0.00%,100.00%
3,100.00%,0.00%
4,100.00%,0.00%


## Persistence and transition meaning

Transitions are estimated on the held-out event-time path. They describe how structural events move between inferred states; they are not price-direction forecasts.

In [5]:
test_states = selected_fit.states["test"]
test_transition = pd.DataFrame(
    transition_matrix(test_states, selected_fit.n_states),
    index=[f"state_{state}" for state in range(selected_fit.n_states)],
    columns=[f"state_{state}" for state in range(selected_fit.n_states)],
)
display(test_transition.style.format("{:.3f}"))

semantic_map = pd.Series({
    state: REGIME_ORDER[regime_code] for state, regime_code in selected_fit.semantic_map.items()
}, name="training_overlap_name")
display(semantic_map.to_frame())

,state_0,state_1,state_2,state_3,state_4
state_0,0.000,0.000,1.000,0.000,0.000
state_1,0.237,0.487,0.000,0.000,0.276
state_2,0.252,0.129,0.619,0.000,0.000
state_3,0.000,0.117,0.000,0.596,0.287
state_4,0.014,0.000,0.000,0.986,0.000


,training_overlap_name
0,broad_coherence
1,deep_counter
2,shallow_counter
3,fragmented
4,broad_coherence


## Later diagnostic behaviour

Forward absolute movement, realized volatility, and multiscale DCC count help name a regime economically. They remain hindsight outcomes and never enter the causal state assignment. Results are shown separately for the chronological test partition.

In [6]:
diagnostics = add_future_diagnostics(global_state, FUTURE_HORIZONS)
future_columns = []
for horizon in FUTURE_HORIZONS:
    future_columns.extend([
        f"future_return_{horizon}",
        f"future_abs_return_{horizon}",
        f"future_realized_vol_{horizon}",
        f"future_dcc_count_{horizon}",
    ])
test_cards = state_path.loc[state_path["partition"].eq("test")].join(
    diagnostics[future_columns], how="left"
)
future_profile = test_cards.groupby("model_state", observed=True)[future_columns].agg(["median", "mean"])
display(future_profile)

directional_check = test_cards.groupby("model_state", observed=True)[
    [f"future_return_{horizon}" for horizon in FUTURE_HORIZONS]
].mean()
display(directional_check.style.format("{:.4%}"))
print("Directional means are diagnostic and do not authorize a directional regime or strategy interpretation.")

future_return_6           future_abs_return_6            \
                     median      mean              median      mean   
model_state                                                           
0                 -0.000089 -0.000067            0.000495  0.000713   
1                  0.000041  0.000009            0.000459  0.000630   
2                  0.000103  0.000066            0.000476  0.000612   
3                 -0.000056 -0.000009            0.000352  0.000466   
4                  0.000113  0.000060            0.000384  0.000566   

            future_realized_vol_6           future_dcc_count_6            \
                           median      mean             median      mean   
model_state                                                                
0                        0.000689  0.000861                0.0  0.107143   
1                        0.000683  0.000783                0.0  0.105263   
2                        0.000568  0.000733                0.0  0.129252   
3                        0.000566  0.000646                0.0  0.127907   
4                        0.000648  0.000769                0.0  0.042254   

            future_return_24            ... future_dcc_count_24            \
                      median      mean  ...              median      mean   
model_state                             ...                                 
0                   0.000217  0.000320  ...                 0.0  0.446429   
1                  -0.000161 -0.000173  ...                 0.0  0.605263   
2                   0.000060  0.000135  ...                 0.0  0.619048   
3                  -0.000034 -0.000025  ...                 0.0  0.529070   
4                   0.000334  0.000334  ...                 0.0  0.436620   

            future_return_96           future_abs_return_96            \
                      median      mean               median      mean   
model_state                                                             
0                  -0.000056  0.000109             0.001840  0.002366   
1                  -0.000328  0.000006             0.001383  0.001991   
2                   0.000120 -0.000168             0.001199  0.001743   
3                  -0.000265  0.000089             0.001449  0.001909   
4                   0.000566  0.000512             0.001385  0.002048   

            future_realized_vol_96           future_dcc_count_96            
                            median      mean              median      mean  
model_state                                                                 
0                         0.002691  0.002947                 2.0  2.142857  
1                         0.002594  0.002820                 2.0  2.144737  
2                         0.002224  0.002606                 1.0  2.136054  
3                         0.002446  0.002624                 2.0  2.111765  
4                         0.002554  0.002784                 1.0  1.957746  

[5 rows x 24 columns]

,future_return_6,future_return_24,future_return_96
model_state,,,
0,-0.0067%,0.0320%,0.0109%
1,0.0009%,-0.0173%,0.0006%
2,0.0066%,0.0135%,-0.0168%
3,-0.0009%,-0.0025%,0.0089%
4,0.0060%,0.0334%,0.0512%


Directional means are diagnostic and do not authorize a directional regime or strategy interpretation.


## Actual price path and filtered confidence

In [7]:
probability_columns = [f"filtered_p_state_{state}" for state in range(selected_fit.n_states)]
plot_price_states(
    state_path,
    "model_semantic",
    f"{SYMBOL}: economically interpreted causal filtered regimes",
    probability_columns=probability_columns,
    last_n=LAST_N,
).show()
plot_price_states(
    state_path,
    "directed_regime",
    f"{SYMBOL}: direct structural comparator on the same observations",
    last_n=LAST_N,
).show()

# Phase 23 completion gate

For each inferred state, be able to state:

1. Its current hierarchy shape and direction balance.
2. Whether it represents agreement, a coherent frontier, or fragmentation.
3. Its scale ages, reversal pressure, and DCC activity.
4. Its filtered confidence and persistence.
5. How it transitions to other states.
6. Which later activity/volatility differences are diagnostic rather than causal inputs.
7. Whether it adds a stable distinction beyond the direct baseline.
8. Why none of this yet specifies a position direction, entry, exit, or expected net return.